# Pokemon Classification with Transfer Learning

**실험 목표:** Pretrained backbone과 fine-tuning 범위가 포켓몬 이미지 분류 성능에 어떤 영향을 주는지 비교.

## 실험 설정 (ResNet18 ablation)

| Exp | Backbone | Pretrained | Fine-tuning 범위 |
|---|---|---|---|
| 1 | ResNet18 | X (scratch) | 전체 학습 |
| 2 | ResNet18 | O | Linear probe (FC만 학습) |
| 3 | ResNet18 | O | Partial (layer4 + FC) |
| 4 | ResNet18 | O | Full fine-tuning |

## 환경
- Google Colab + GPU (T4 권장)
- 데이터셋: `lantian773030/pokemonclassification` (Kaggle, 약 7,000장 / 150 클래스)
- 전처리: Resize 224×224 + ImageNet mean/std normalize (data augmentation 미사용)
- 공통: AdamW, weight decay 1e-4, CosineAnnealingLR, label smoothing 0.1, batch 32, seed 42
- Split: train/val/test = 70/15/15 (stratified)

## 1. 환경 준비

In [ ]:
!pip install -q kagglehub

In [ ]:
import os, time, random, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix
)

import kagglehub

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

## 2. 데이터셋 다운로드 및 로드

In [ ]:
path = kagglehub.dataset_download('lantian773030/pokemonclassification')
print('Dataset path:', path)

In [ ]:
# ImageFolder 형식 (클래스 폴더가 다수 있는 디렉토리) 자동 탐색
def find_data_root(base):
    for root, dirs, _ in os.walk(base):
        subdirs = [d for d in dirs if not d.startswith('.')]
        if len(subdirs) >= 10:
            sample_dir = os.path.join(root, subdirs[0])
            files_in_sample = os.listdir(sample_dir)
            if any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in files_in_sample):
                return root
    return base

data_root = find_data_root(path)
print('Data root:', data_root)
print('하위 클래스 폴더 수:', sum(1 for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))))

In [ ]:
# 전처리 (augmentation 없음, Resize + ImageNet normalize만)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_dataset = datasets.ImageFolder(data_root, transform=transform)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f'전체 이미지 수: {len(full_dataset)}')
print(f'클래스 수: {num_classes}')
print('클래스 예시:', class_names[:8])

In [ ]:
# Stratified split 70/15/15
labels = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

train_idx, temp_idx, _, temp_y = train_test_split(
    indices, labels, test_size=0.30, stratify=labels, random_state=SEED
)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, temp_y, test_size=0.50, stratify=temp_y, random_state=SEED
)

train_ds = Subset(full_dataset, train_idx)
val_ds = Subset(full_dataset, val_idx)
test_ds = Subset(full_dataset, test_idx)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

## 3. 모델 정의

각 실험마다 ResNet18을 다른 방식으로 구성. `requires_grad`로 fine-tuning 범위를 제어.

In [ ]:
def build_model(experiment, num_classes):
    if experiment == 1:
        # Scratch (no pretrained), 전체 학습
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif experiment == 2:
        # Pretrained + linear probe (backbone 동결)
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for p in model.parameters():
            p.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)  # 새 FC만 학습
    elif experiment == 3:
        # Pretrained + partial (layer4 + FC만 학습)
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for p in model.parameters():
            p.requires_grad = False
        for p in model.layer4.parameters():
            p.requires_grad = True
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif experiment == 4:
        # Pretrained + full fine-tuning
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(experiment)
    return model

EXP_CONFIG = {
    1: {'name': 'ResNet18-scratch',      'epochs': 30, 'lr': 1e-3},
    2: {'name': 'ResNet18-linear-probe', 'epochs': 15, 'lr': 1e-3},
    3: {'name': 'ResNet18-partial',      'epochs': 15, 'lr': 1e-4},
    4: {'name': 'ResNet18-full-FT',      'epochs': 15, 'lr': 1e-4},
}

## 4. 학습 / 평가 루틴

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, total_correct / total

@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        if criterion is not None:
            total_loss += criterion(out, y).item() * x.size(0)
        preds = out.argmax(1)
        total_correct += (preds == y).sum().item()
        total += x.size(0)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(y.cpu().numpy())
    avg_loss = (total_loss / total) if criterion is not None else None
    return avg_loss, total_correct / total, np.concatenate(all_preds), np.concatenate(all_labels)

In [ ]:
def run_experiment(exp_id, patience=5):
    cfg = EXP_CONFIG[exp_id]
    print(f"\n=== Exp {exp_id}: {cfg['name']} | epochs={cfg['epochs']} | lr={cfg['lr']} ===")
    set_seed(SEED)

    model = build_model(exp_id, num_classes).to(DEVICE)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=cfg['lr'], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, best_state, no_improve = 0.0, None, 0

    for epoch in range(1, cfg['epochs'] + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        print(f"  ep{epoch:02d}/{cfg['epochs']} | train {tr_loss:.3f}/{tr_acc:.3f} | val {val_loss:.3f}/{val_acc:.3f} | {time.time()-t0:.1f}s")

        if no_improve >= patience:
            print(f'  Early stopping at epoch {epoch} (no improvement for {patience} epochs)')
            break

    # Best weight 로드 후 test 평가
    model.load_state_dict(best_state)
    _, test_acc, test_preds, test_labels = evaluate(model, test_loader)
    test_prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
    test_rec = recall_score(test_labels, test_preds, average='macro', zero_division=0)
    test_f1 = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    print(f'  [Test] acc={test_acc:.4f} | prec={test_prec:.4f} | rec={test_rec:.4f} | F1={test_f1:.4f}')

    out = {
        'name': cfg['name'],
        'history': history,
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'test_prec': test_prec,
        'test_rec': test_rec,
        'test_f1': test_f1,
        'test_preds': test_preds,
        'test_labels': test_labels,
        'model_state': best_state,
    }
    del model
    torch.cuda.empty_cache()
    return out

## 5. 4개 실험 실행

In [ ]:
results = {}
for exp_id in [1, 2, 3, 4]:
    results[exp_id] = run_experiment(exp_id)

## 6. 결과 비교

### 6-1. Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {1: 'tab:gray', 2: 'tab:blue', 3: 'tab:orange', 4: 'tab:green'}

for exp_id, res in results.items():
    h = res['history']
    epochs = range(1, len(h['train_loss']) + 1)
    c = colors[exp_id]
    axes[0].plot(epochs, h['train_loss'], color=c, alpha=0.4, linestyle='--')
    axes[0].plot(epochs, h['val_loss'], color=c, label=f"Exp{exp_id} {res['name']}")
    axes[1].plot(epochs, h['train_acc'], color=c, alpha=0.4, linestyle='--')
    axes[1].plot(epochs, h['val_acc'], color=c, label=f"Exp{exp_id} {res['name']}")

axes[0].set_title('Loss (dashed=train, solid=val)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
axes[1].set_title('Accuracy (dashed=train, solid=val)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()

### 6-2. Test metrics 표

In [ ]:
summary = pd.DataFrame([
    {
        'Experiment': f'Exp {eid}',
        'Model': res['name'],
        'Test Accuracy': round(res['test_acc'], 4),
        'Test Precision': round(res['test_prec'], 4),
        'Test Recall': round(res['test_rec'], 4),
        'F1-score': round(res['test_f1'], 4),
    }
    for eid, res in results.items()
])
summary.to_csv('results_summary.csv', index=False)
summary

### 6-3. Confusion matrix top-N (best 모델 기준)

150×150 전체는 시각화가 어려워 가장 자주 발생하는 오분류와 recall이 낮은 클래스만 표로 출력.

In [ ]:
best_exp = max(results.keys(), key=lambda k: results[k]['test_acc'])
print(f"Best experiment: Exp {best_exp} ({results[best_exp]['name']}) — test acc {results[best_exp]['test_acc']:.4f}")

cm = confusion_matrix(
    results[best_exp]['test_labels'],
    results[best_exp]['test_preds'],
    labels=list(range(num_classes))
)

N = 15

# (1) 가장 자주 발생한 오분류 pair
misclass = []
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm[i, j] > 0:
            misclass.append((int(cm[i, j]), class_names[i], class_names[j]))
misclass.sort(reverse=True)
print(f'\nTop-{N} 오분류 (true → predicted):')
top_mis_df = pd.DataFrame(misclass[:N], columns=['count', 'true', 'predicted'])
print(top_mis_df.to_string(index=False))

In [ ]:
# (2) Recall이 가장 낮은 클래스
class_recall = []
for i in range(num_classes):
    n = int(cm[i, :].sum())
    if n > 0:
        class_recall.append((cm[i, i] / n, class_names[i], n))
class_recall.sort()
print(f'Worst-{N} class recall:')
worst_df = pd.DataFrame(
    [(round(r, 3), c, n) for r, c, n in class_recall[:N]],
    columns=['recall', 'class', 'n_test']
)
print(worst_df.to_string(index=False))

## 7. Best 모델 저장 (Streamlit 데모용)

`best_model.pt`을 다운로드해서 `app.py`와 같은 폴더에 두고 실행하면 GUI 데모를 사용할 수 있음.

In [ ]:
torch.save({
    'model_state': results[best_exp]['model_state'],
    'class_names': class_names,
    'experiment': best_exp,
    'experiment_name': results[best_exp]['name'],
    'num_classes': num_classes,
    'img_size': IMG_SIZE,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
}, 'best_model.pt')
print('Saved: best_model.pt')
print('Colab에서 좌측 파일 패널 → best_model.pt 우클릭 → 다운로드')